<h1 style="text-align: center;"> CVE Trends and Characteristics Analysis (2022–2025)

</h1>

Collaborators: Jason Pang, Jimmy Zhang, Neal Harricharan, Praneel Jakkampudi
<hr>

## Introduction

The CVE dataset contains a large amount of cybersecurity data including for most of the entries metrics relating to what kind of vulnerability it is, how the vulnerability is exploited, what it impacts, and the cvss severity score. 

The overall objective of this analysis is to investigate the distribution and structural trends within Common Vulnerabilities and Exposures (CVE) between 2022 and 2025. Throughout this tutorial, this report will uncovers trends across the years and visualizing what kinds of vulnerabilities and severities are most common.

Over this tutorial, we will be going through the Data Science Lifecycle as follows:

- [Data Collection](#data-collection)
- [Data Processing](#data-processing)
- [Exploratory Analysis & Data Visualization](#exploratory-analysis--data-visualization)
- ~~[Analysis, Hypothesis Testing, & ML](#analysis-hypothesis-testing--ml)~~
- ~~[Insight & Policy Decision](#insight--policy-decision)~~

## Data Collection

During data collection stage, we mainly focus on collecting data from websites or various files. For this report, we directly gathered the data from [CVEProject/cvelistV5 Github Repository](https://github.com/CVEProject/cvelistv5), the cache of the official CVE List in CVE JSON 5 format. 

```bash
# TODO COMMAND FOR IMPORTING
```


### Tools Used

We used [Pandas](https://pandas.pydata.org/), an open-source Python library, as a primary tool to manipulate the data. Due to the large size of the data across many years, we parse and normalize each report's JSON data before storing in a [SQLite](https://www.sqlite.org/) database for better data handling during data processing. In later phases, we use [NumPy](https://numpy.org/) to perform fast calculations and analysis.

In [ ]:
# For Data Processing
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# For Data Visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.ticker import PercentFormatter

# Global Configuration
DATA_DIRECTORY = Path("./cves")             # Path to the directory containing CVE JSON files
DATABASE_PATH = Path("./database.sqlite")   # Path to the SQLite database file
YEARS = [2022, 2023, 2024, 2025]            # Years to process CVE data for

## Data Processing

### Part 1: Converting to SQLite

First, we flatten the JSON data into a SQLite database. You can skip to part 2 if [database.sqlite](#part-2-load-and-reformat-sqlite-as-pandas-dataframe) is already created.

In [ ]:
# Local Data Processing Variables (del later to avoid namespace pollution)
# Values that represent missing data and the replacement used for each field.
EMPTY_VALUES = {"", "NULL", "N/A", "NA", "UNKNOWN", "NONE"}
MISSING_VALUE_REPLACEMENTS = {
    "product": None,
    "vendor": None,
    "availabilityImpact": "None",
    "confidentialityImpact": "None",
    "integrityImpact": "None",
    "privilegesRequired": "None",
    "userInteraction": "None",
    "scope": "Unchanged",
}

# Constants
COLUMNS = [
    "cveId",
    "dataVersion",
    "datePublished",
    "product",
    "vendor",
    "cweId",
    "attackComplexity",
    "attackVector",
    "availabilityImpact",
    "baseScore",
    "baseSeverity",
    "confidentialityImpact",
    "integrityImpact",
    "privilegesRequired",
    "scope",
    "userInteraction",
]
STRING_COLUMNS = [column for column in COLUMNS if not(column == "baseScore" or column == "datePublished" or column == "cveId" or column == "dataVersion")]
FLOAT_COLUMNS = ["dataVersion", "baseScore"]
VECTOR_KEY_TRANSLATION = {
    "AV": "attackVector",
    "AC": "attackComplexity",
    "PR": "privilegesRequired",
    "UI": "userInteraction",
    "S": "scope",
    "C": "confidentialityImpact",
    "I": "integrityImpact",
    "A": "availabilityImpact",
    "VC": "confidentialityImpact",
    "VI": "integrityImpact",
    "VA": "availabilityImpact",
}
VECTOR_VALUE_TRANSLATION = {
    "attackVector": {"N": "Network", "A": "Adjacent", "L": "Local", "P": "Physical"},
    "userInteraction": {"N": "None", "R": "Required", "P": "Passive", "A": "Active"},
}
DEFAULT_VECTOR_VALUES = {
    "N": "None",
    "L": "Low",
    "P": "Partial",
    "C": "Changed",
    "M": "Medium",
    "H": "High",
    "U": "Unchanged",
    "R": "Required",
}
KEY_REPLACEMENTS = {
    "vulnAvailabilityImpact": "availabilityImpact",
    "vulnConfidentialityImpact": "confidentialityImpact",
    "vulnIntegrityImpact": "integrityImpact",
}
BASE_SEVERITY_BREAKPOINTS = (
    (4.0, "Low"),
    (7.0, "Medium"),
    (9.0, "High"),
    (10.1, "Critical"),
)
REQUIRED_FIELDS = {
    "cveId": "missing or empty CVE identifier",
    "datePublished": "missing or empty publication date",
    "baseScore": "missing or empty base score",
}
ATTACK_VECTOR_REPLACEMENTS = {
    "Adjacent": "Adjacent",
    "Adjacent network": "Adjacent",
}
BASE_SEVERITY_REPLACEMENTS = {
    "None": "None",
    "Low": "Low",
    "Medium": "Medium",
    "Moderate": "Medium",
    "High": "High",
    "Critical": "Critical",
}
USER_INTERACTION_REPLACEMENTS = {
    "None": "None",
    "Required": "Required",
    "Active": "Required",
    "Passive": "Required",
}

In [ ]:
# Custom exception for dropped records
class DroppedRecordError(ValueError):
    """Raised when a CVE record cannot be inserted into the database."""

In [ ]:
# Parse the nested JSON structure into a flat dictionary of key/value pairs.
def flatten_value(key: str, value: Any, target: dict[str, Any]) -> None:
    """Flatten nested JSON values into the key/value representation used here."""
    if isinstance(value, dict):
        flatten_dict(value, target)
    elif isinstance(value, list):
        if not value:
            target[key] = ""
        elif len(value) == 1:
            flatten_value(key, value[0], target)
        else:
            for index, item in enumerate(value):
                flatten_value(f"{key}({index})", item, target)
    else:
        if (not (key in target.keys())):
            target[key] = value

def flatten_dict(source: dict[str, Any], target: dict[str, Any]) -> None:
    """Flatten every value in a CVE JSON object into ``target``."""
    for key, value in source.items():
        flatten_value(key, value, target)

In [ ]:
# Since vector string are more reliable than the individual vector components (records),
# we overwrite the individual vector components with the values from the vector string.
def translate_vector(vector_string: str, target: dict[str, Any]) -> None:
    """Decode CVSS vector abbreviations and add their values to ``target``."""
    # Check if CVSS verison is a supported version
    vector_list = vector_string.split("/")
    short_key, value = vector_list[0].split(":", 1)

    if short_key == "CVSS":
        match value:
            case "3.0" | "3.1":
                for component in vector_list[1:]:
                    if ":" not in component:
                        continue
                    short_key, value = component.split(":", 1)
                    key = VECTOR_KEY_TRANSLATION.get(short_key)
                    if key is None:
                        continue
                    target[key] = VECTOR_VALUE_TRANSLATION.get(key, DEFAULT_VECTOR_VALUES).get(
                        value, value
                    )
            case "4.0" | "4.1":
                scope = False
                for component in vector_list[1:]:
                    if ":" not in component:
                        continue
                    short_key, value = component.split(":", 1)
                    key = VECTOR_KEY_TRANSLATION.get(short_key)
                    if(short_key in ["SC", "SI", "SA"] and value != 'N'):
                        scope = True
                    if key is None:
                        continue
                    target[key] = VECTOR_VALUE_TRANSLATION.get(key, DEFAULT_VECTOR_VALUES).get(
                        value, value
                    )
                if scope:
                    target['scope'] = 'Changed'
                else:
                    target['scope'] = 'Unchanged'
            case _:
                raise DroppedRecordError(f"CVSS version {value} is not recognized") 
    else:
        raise DroppedRecordError(f"CVSS != {short_key}")

In [ ]:
# Replace missing or empty values with configured defaults.
def is_empty_value(value: Any) -> bool:
    """Return whether a value represents missing data."""
    return value is None or (
        isinstance(value, str) and value.strip().upper() in EMPTY_VALUES
    )

def replace_missing_value(
    key: str, record: dict[str, Any], flattened: dict[str, Any]
) -> Any:
    """Return the configured replacement for a missing or empty field."""
    if key == "baseSeverity":
        score = flattened.get("baseScore")
        if score is not None and not is_empty_value(score):
            if score == 0:
                return "None"
            for breakpoint, severity in BASE_SEVERITY_BREAKPOINTS:
                if score < breakpoint:
                    return severity
            raise ValueError(f"base score is outside the CVSS range: {score}")
    return MISSING_VALUE_REPLACEMENTS.get(key)

In [ ]:
# Normalization functions for string values, numeric values, and publication timestamps.
def normalize_string_values(record: dict[str, Any]) -> None:
    """Capitalize string columns and standardize attack-vector labels."""
    # Non-identifier string columns are capitalized for consistency
    for column in STRING_COLUMNS:
        value = record[column]
        if isinstance(value, str):
            record[column] = value.strip().capitalize()

    # Adjacent Network == Adjacent so...
    attack_vector = record["attackVector"]
    if isinstance(attack_vector, str):
        record["attackVector"] = ATTACK_VECTOR_REPLACEMENTS.get(
            attack_vector, attack_vector
        )

    # Moderate == Medium so...
    base_severity = record["baseSeverity"]
    if isinstance(base_severity, str):
        record["baseSeverity"] = BASE_SEVERITY_REPLACEMENTS.get(
            base_severity, base_severity
        )

    # Active and Passive == Required so...
    user_interaction = record["userInteraction"]
    if isinstance(user_interaction, str):
        record["userInteraction"] = USER_INTERACTION_REPLACEMENTS.get(
            user_interaction, user_interaction
        )


def normalize_date(value: Any) -> str:
    """Normalize an ISO-8601 publication date to a UTC timestamp with time."""
    if not isinstance(value, str):
        raise DroppedRecordError("datePublished is not a valid timestamp")
    try:
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError as error:
        raise DroppedRecordError("datePublished is not a valid timestamp") from error
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc).isoformat(timespec="microseconds").replace(
        "+00:00", "Z"
    )


def normalize_special_values(record: dict[str, Any]) -> None:
    """Normalize identifier, numeric fields, and publication timestamps."""
    # Normalize identifier columns (all uppercase)
    for column in ("cveId", "cweId"):
        value = record[column]
        if isinstance(value, str):
            record[column] = value.upper()

    # Normalize numeric columns
    for column in FLOAT_COLUMNS:
        value = record[column]
        if not is_empty_value(value):
            record[column] = float(value)

    # Normalize the publication date to a UTC timestamp
    if not is_empty_value(record["datePublished"]):
        record["datePublished"] = normalize_date(record["datePublished"])

In [ ]:
# Parse one CVE record from the CVE JSON into a flat dictionary of key/value pairs.
def parse_record(data: dict[str, Any]) -> dict[str, Any]:
    """Flatten, rename, translate, and validate one CVE record."""
    flattened: dict[str, Any] = {}
    flatten_dict(data, flattened)
    for old_key, new_key in KEY_REPLACEMENTS.items():
        if old_key in flattened and new_key not in flattened:
            flattened[new_key] = flattened[old_key]

    record: dict[str, Any] = {}
    for key in COLUMNS:
        value = flattened.get(key)
        record[key] = (
            replace_missing_value(key, record, flattened)
            if is_empty_value(value)
            else value
        )

    vector = flattened.get("vectorString")
    if not is_empty_value(vector):
        translate_vector(str(vector), record)

    normalize_string_values(record)
    normalize_special_values(record)

    for key, error_message in REQUIRED_FIELDS.items():
        if is_empty_value(record[key]):
            raise DroppedRecordError(error_message)
    record["baseScore"] = float(record["baseScore"])
    return record

In [ ]:
# Load and create the SQLite database with per-year tables and insert the CVE records.
def create_tables(connection: sqlite3.Connection) -> None:
    """Create the per-year tables with the shared CVE schema."""
    column_sql = ", ".join(
        f"{column} {'REAL' if column in FLOAT_COLUMNS else 'TEXT'}"
        f"{' PRIMARY KEY' if column == 'cveId' else ''}"
        for column in COLUMNS
    )
    for year in YEARS:
        table = f"YEAR_{year}"
        connection.execute(f"DROP TABLE IF EXISTS {table}")
        connection.execute(f"CREATE TABLE {table} ({column_sql})")


def load_records(connection: sqlite3.Connection) -> list[tuple[str, str]]:
    """Load records into SQLite and return CVE IDs and validation errors."""
    dropped: list[tuple[str, str]] = []
    placeholders = ", ".join("?" for _ in COLUMNS)
    columns = ", ".join(COLUMNS)
    for path in DATA_DIRECTORY.rglob("*.json"):
        with path.open(encoding="utf-8") as file:
            source = json.load(file)
        cve_id = str(source.get("cveMetadata", {}).get("cveId", path.stem))
        try:
            record = parse_record(source)
        except DroppedRecordError as error:
            dropped.append((cve_id, str(error)))
            continue
        year = path.parts[-3]
        values = [record[column] for column in COLUMNS]
        connection.execute(
            f"INSERT INTO YEAR_{year} ({columns}) VALUES ({placeholders})", values
        )
    return dropped

In [ ]:
# Build the SQLite database and log records excluded during normalization.
with sqlite3.connect(DATABASE_PATH) as connection:
    create_tables(connection)
    dropped_data = load_records(connection)

print(f"Loaded CVE data into {DATABASE_PATH}")

In [ ]:
# Summarize the reasons for dropped records dataframe
dropped = pd.DataFrame(dropped_data)

# Summarize the reasons for dropped records dataframe
display(dropped.groupby(1).size().reset_index(name='count').rename(columns={1: 'Reason'}).sort_values(by='count', ascending=False))

# Show example dropped records for each unique error
display(dropped.drop_duplicates(subset=1).rename(columns={0: 'CVE ID', 1: 'Reason'}))

In [ ]:
# Clearing the dropped records dataframe and the DroppedRecordError class to reduce memory usage
del ( dropped_data, dropped, DroppedRecordError )

# Clear the data processing variables to avoid namespace pollution
del ( EMPTY_VALUES, MISSING_VALUE_REPLACEMENTS, COLUMNS, STRING_COLUMNS, FLOAT_COLUMNS, VECTOR_KEY_TRANSLATION, VECTOR_VALUE_TRANSLATION, DEFAULT_VECTOR_VALUES, KEY_REPLACEMENTS, BASE_SEVERITY_BREAKPOINTS, REQUIRED_FIELDS, ATTACK_VECTOR_REPLACEMENTS, BASE_SEVERITY_REPLACEMENTS, USER_INTERACTION_REPLACEMENTS )

# Clear data processing functions to avoid namespace pollution
del ( flatten_value, flatten_dict, translate_vector, replace_missing_value, normalize_string_values, normalize_date, normalize_special_values, parse_record, create_tables, load_records )

# Uncomment the following line to display all variables in the current namespace after data processing is complete
%whos

## Exploratory Analysis & Data Visualization

> **TODO: CHECKPOINT 2**

In [ ]:
# Local Exploratory Analysis variables (del later to avoid namespace pollution)
OUTPUT_DIR = Path("cve_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# Fetch all records from the per-year tables into a single DataFrame for analysis
if not DATABASE_PATH.is_file():
    raise FileNotFoundError("Place database.sqlite in same directory as this notebook before running it.")

query = " UNION ALL ".join([f"SELECT *, {year} AS cve_year FROM YEAR_{year}" for year in YEARS])
with sqlite3.connect(DATABASE_PATH) as connection:
    df = pd.read_sql_query(query, connection, parse_dates=["datePublished"])

df.info()  # Display the DataFrame information to verify the data has been loaded correctly

## Analysis, Hypothesis Testing, & ML
*TODO: CHECKPOINT 3*

## Insight & Policy Decision
*TODO: CHECKPOINT 3*